# Personal AI Agent — Upload to GitHub from Colab
Upload the ZIP, enter the repository name, and provide a fine-grained GitHub token with **Contents: read/write** for that repository. Do not save or share the token. This does not deploy hosting or enter API keys.

In [ ]:
# Upload all project files from the ZIP into an existing GitHub repository.
# GitHub token is requested securely and is never written to disk or notebook outputs.
from google.colab import files
from getpass import getpass
from pathlib import Path
import zipfile, tempfile, requests, base64

print('Select Personal_AI_Agent_One_Upload_Deployment_Kit.zip')
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
if not zip_name:
    raise ValueError('Upload the project ZIP first.')
repo = input('GitHub repository (owner/repo): ').strip()
if len(repo.split('/')) != 2:
    raise ValueError('Use owner/repository, for example Atrayee321/Personal-AI-Agent')
token = getpass('GitHub fine-grained token (Contents: read/write): ').strip()
if not token:
    raise ValueError('GitHub token required')
headers = {'Authorization': f'Bearer {token}', 'Accept':'application/vnd.github+json', 'X-GitHub-Api-Version':'2022-11-28'}
base = f'https://api.github.com/repos/{repo}/contents/'
check = requests.get(f'https://api.github.com/repos/{repo}', headers=headers, timeout=30)
check.raise_for_status()
branch = check.json()['default_branch']
with tempfile.TemporaryDirectory() as temp:
    with zipfile.ZipFile(zip_name) as archive:
        members = [m for m in archive.infolist() if not m.is_dir()]
        # Avoid unsafe zip paths; no files outside temp are extracted.
        paths = [Path(m.filename) for m in members]
        if any(p.is_absolute() or '..' in p.parts for p in paths):
            raise ValueError('Unsafe archive paths')
        prefix = paths[0].parts[0] if paths and all(p.parts[0] == paths[0].parts[0] for p in paths) else None
        count = 0
        for member, p in zip(members, paths):
            relative = Path(*p.parts[1:]) if prefix else p
            if not relative.parts or relative.name in {'.env','secrets.toml'}:
                continue
            path = relative.as_posix()
            url = base + '/'.join(requests.utils.quote(x, safe='') for x in relative.parts)
            existing = requests.get(url, headers=headers, params={'ref':branch}, timeout=30)
            if existing.status_code not in (200,404):
                existing.raise_for_status()
            data = {'message': f'Upload project file: {path}', 'branch':branch,
                    'content':base64.b64encode(archive.read(member)).decode()}
            if existing.status_code == 200:
                data['sha'] = existing.json()['sha']
            response = requests.put(url, headers=headers, json=data, timeout=60)
            if response.status_code not in (200,201):
                raise RuntimeError(f'Upload failed for {path}: HTTP {response.status_code} {response.text[:250]}')
            count += 1
print(f'Success: uploaded {count} files to https://github.com/{repo} on branch {branch}')
print('Next: Deploy Streamlit with main file streamlit_app.py and add secrets in Streamlit Cloud.')
print('GitHub Actions will build the Android debug APK. Check Actions > Build Android APK.')
